In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import random
import json
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding
from tqdm.auto import tqdm

from sklearn.metrics import accuracy_score

In [2]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

PyTorch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA: 12.1


device(type='cuda')

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
ann = pd.read_csv("../data/processed/mhs_main_experiment_annotations_with_split.csv")

print("Annotation rows:", ann.shape)
print("Unique comments:", ann["comment_id"].nunique())
print(ann["split"].value_counts())

Annotation rows: (49433, 53)
Unique comments: 19761
split
train         34413
test           7713
validation     7307
Name: count, dtype: int64


In [5]:
LABELS = [0, 1, 2]
label_cols = [f"hatespeech_{label}_prob" for label in LABELS]

def make_distribution(group):
    counts = group["hatespeech"].value_counts(normalize=True)
    return pd.Series({
        f"hatespeech_{label}_prob": counts.get(label, 0.0)
        for label in LABELS
    })

def entropy_np(probs):
    probs = np.array(probs)
    probs = probs[probs > 0]
    if len(probs) == 0:
        return 0.0
    return -np.sum(probs * np.log2(probs))

In [6]:
metadata = (
    ann.groupby("comment_id")
    .agg(
        text_clean=("text_clean", "first"),
        split=("split", "first"),
        target_type=("target_type", "first"),
        is_women_targeted=("is_women_targeted", "max"),
        is_immigrant_targeted=("is_immigrant_targeted", "max"),
        annotation_count=("annotator_id", "count"),
        unique_annotators=("annotator_id", "nunique")
    )
    .reset_index()
)

dist_df = (
    ann.groupby("comment_id")
    .apply(lambda g: make_distribution(g))
    .reset_index()
)

comment_df = metadata.merge(dist_df, on="comment_id", how="left")

comment_df["hatespeech_entropy"] = comment_df[label_cols].apply(
    lambda row: entropy_np(row.values),
    axis=1
)

comment_df["hatespeech_majority"] = comment_df[label_cols].values.argmax(axis=1)

print("Comment-level shape:", comment_df.shape)
comment_df.head()

Comment-level shape: (19761, 13)


/tmp/ipykernel_11481/1404436087.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: make_distribution(g))


,comment_id,text_clean,split,target_type,is_women_targeted,is_immigrant_targeted,annotation_count,unique_annotators,hatespeech_0_prob,hatespeech_1_prob,hatespeech_2_prob,hatespeech_entropy,hatespeech_majority
0,6,First off you look cool as fuck! Anyway if we ...,train,women_only,1,0,2,2,1.000000,0.0,0.000000,-0.000000,0
1,7,\*points to posters asking for palestinian rig...,test,immigrant_only,0,1,2,2,0.500000,0.0,0.500000,1.000000,0
2,10,"They'll come back in your plan, also. Plus we ...",train,immigrant_only,0,1,3,3,0.333333,0.0,0.666667,0.918296,2
3,11,"eat my fuck, bitch",validation,women_only,1,0,2,2,0.000000,0.5,0.500000,1.000000,1
4,12,She ugly anyways,train,women_only,1,0,2,2,0.500000,0.0,0.500000,1.000000,0


In [7]:
MIN_ANNOTATIONS = 2

reliable_df = comment_df[
    comment_df["annotation_count"] >= MIN_ANNOTATIONS
].copy()

print("Original comments:", len(comment_df))
print("Reliable comments:", len(reliable_df))
print("Percent kept:", round(len(reliable_df) / len(comment_df) * 100, 2))

Original comments: 19761
Reliable comments: 10639
Percent kept: 53.84


In [9]:
full_split_counts = (
    comment_df.groupby("split")
    .agg(
        comments=("comment_id", "count"),
        mean_annotation_count=("annotation_count", "mean"),
        mean_entropy=("hatespeech_entropy", "mean"),
        zero_entropy_percent=("hatespeech_entropy", lambda x: round((x == 0).mean() * 100, 2))
    )
    .reset_index()
)

reliable_split_counts = (
    reliable_df.groupby("split")
    .agg(
        comments=("comment_id", "count"),
        mean_annotation_count=("annotation_count", "mean"),
        mean_entropy=("hatespeech_entropy", "mean"),
        zero_entropy_percent=("hatespeech_entropy", lambda x: round((x == 0).mean() * 100, 2))
    )
    .reset_index()
)

print("Full split counts:")
display(full_split_counts)

print("Reliable subset split counts:")
display(reliable_split_counts)

Full split counts:


,split,comments,mean_annotation_count,mean_entropy,zero_entropy_percent
0,test,2965,2.601349,0.228469,77.30
1,train,13832,2.487927,0.223532,77.56
2,validation,2964,2.465250,0.228493,77.13


Reliable subset split counts:


,split,comments,mean_annotation_count,mean_entropy,zero_entropy_percent
0,test,1601,3.965646,0.423117,57.96
1,train,7437,3.767379,0.415744,58.26
2,validation,1601,3.712680,0.423018,57.65


In [10]:
train_entropy = reliable_df.loc[
    reliable_df["split"] == "train",
    "hatespeech_entropy"
]

nonzero_train_entropy = train_entropy[train_entropy > 0]

LOW_THRESHOLD = nonzero_train_entropy.quantile(0.33)
HIGH_THRESHOLD = nonzero_train_entropy.quantile(0.66)

print("Low entropy threshold:", LOW_THRESHOLD)
print("High entropy threshold:", HIGH_THRESHOLD)

Low entropy threshold: 0.9182958340544896
High entropy threshold: 1.0


In [11]:
def assign_entropy_group(ent):
    if ent == 0:
        return "zero_entropy"
    elif ent <= LOW_THRESHOLD:
        return "low_entropy"
    elif ent <= HIGH_THRESHOLD:
        return "medium_entropy"
    else:
        return "high_entropy"

reliable_df["entropy_group"] = reliable_df["hatespeech_entropy"].apply(assign_entropy_group)

entropy_group_summary = (
    reliable_df
    .groupby(["split", "entropy_group"])
    .agg(
        comments=("comment_id", "count"),
        mean_entropy=("hatespeech_entropy", "mean"),
        mean_annotation_count=("annotation_count", "mean")
    )
    .reset_index()
    .sort_values(["split", "entropy_group"])
)

entropy_group_summary

,split,entropy_group,comments,mean_entropy,mean_annotation_count
0,test,high_entropy,66,1.531993,7.000000
1,test,low_entropy,276,0.888765,10.032609
2,test,medium_entropy,331,1.000000,2.151057
3,test,zero_entropy,928,0.000000,2.592672
4,train,high_entropy,252,1.533318,17.293651
5,train,low_entropy,1301,0.887566,7.149116
6,train,medium_entropy,1551,0.999850,2.367505
7,train,zero_entropy,4333,0.000000,2.466420
8,validation,high_entropy,57,1.535602,15.526316
9,validation,low_entropy,277,0.887087,7.386282


In [14]:
ENTROPY_WEIGHTS = {
    "zero_entropy": 0.75,
    "low_entropy": 1.25,
    "medium_entropy": 1.5,
    "high_entropy": 2.0,
}

reliable_df["sample_weight"] = reliable_df["entropy_group"].map(ENTROPY_WEIGHTS)

In [15]:
train_df = reliable_df[reliable_df["split"] == "train"].copy()
val_df = reliable_df[reliable_df["split"] == "validation"].copy()
test_df = reliable_df[reliable_df["split"] == "test"].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain entropy groups:")
print(train_df["entropy_group"].value_counts())

print("\nValidation entropy groups:")
print(val_df["entropy_group"].value_counts())

print("\nTest entropy groups:")
print(test_df["entropy_group"].value_counts())

Train: (7437, 15)
Validation: (1601, 15)
Test: (1601, 15)

Train entropy groups:
entropy_group
zero_entropy      4333
medium_entropy    1551
low_entropy       1301
high_entropy       252
Name: count, dtype: int64

Validation entropy groups:
entropy_group
zero_entropy      923
medium_entropy    344
low_entropy       277
high_entropy       57
Name: count, dtype: int64

Test entropy groups:
entropy_group
zero_entropy      928
medium_entropy    331
low_entropy       276
high_entropy       66
Name: count, dtype: int64


In [16]:
hf_cols = [
    "comment_id",
    "text_clean",
    "hatespeech_0_prob",
    "hatespeech_1_prob",
    "hatespeech_2_prob",
    "hatespeech_entropy",
    "entropy_group",
    "sample_weight"
]

train_dataset = Dataset.from_pandas(train_df[hf_cols].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[hf_cols].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[hf_cols].reset_index(drop=True))

In [17]:
model_name = "roberta-base"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [18]:
def tokenize(batch):
    return tokenizer(
        batch["text_clean"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/7437 [00:00<?, ? examples/s]

Map:   0%|          | 0/1601 [00:00<?, ? examples/s]

Map:   0%|          | 0/1601 [00:00<?, ? examples/s]

In [20]:
def add_labels(batch):
    n = len(batch["text_clean"])

    batch["labels"] = [
        [
            float(batch["hatespeech_0_prob"][i]),
            float(batch["hatespeech_1_prob"][i]),
            float(batch["hatespeech_2_prob"][i])
        ]
        for i in range(n)
    ]

    batch["sample_weight"] = [
        float(batch["sample_weight"][i])
        for i in range(n)
    ]

    return batch

train_dataset = train_dataset.map(add_labels, batched=True)
val_dataset = val_dataset.map(add_labels, batched=True)
test_dataset = test_dataset.map(add_labels, batched=True)

Map:   0%|          | 0/7437 [00:00<?, ? examples/s]

Map:   0%|          | 0/1601 [00:00<?, ? examples/s]

Map:   0%|          | 0/1601 [00:00<?, ? examples/s]

In [21]:
columns_to_keep = [
    "input_ids",
    "attention_mask",
    "labels",
    "sample_weight"
]

train_dataset.set_format(type="torch", columns=columns_to_keep)
val_dataset.set_format(type="torch", columns=columns_to_keep)
test_dataset.set_format(type="torch", columns=columns_to_keep)

In [22]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

In [23]:
class HatespeechLeWiDiModel(nn.Module):
    def __init__(self, model_name, dropout_prob=0.35, freeze_layers=4):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        if freeze_layers > 0:
            for param in self.encoder.embeddings.parameters():
                param.requires_grad = False

            for layer in self.encoder.encoder.layer[:freeze_layers]:
                for param in layer.parameters():
                    param.requires_grad = False

        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(hidden_size, 3)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled = outputs.last_hidden_state[:, 0, :]
        pooled = self.dropout(pooled)

        logits = self.classifier(pooled)
        return logits

In [24]:
model = HatespeechLeWiDiModel(
    model_name=model_name,
    dropout_prob=0.35,
    freeze_layers=4
).to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("Trainable params:", trainable_params)
print("Total params:", total_params)
print("Trainable percent:", round(trainable_params / total_params * 100, 2))

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable params: 57295875
Total params: 124647939
Trainable percent: 45.97


In [25]:
def compute_loss(model, batch):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].float().to(device)
    weights = batch["sample_weight"].float().to(device)

    logits = model(input_ids=input_ids, attention_mask=attention_mask)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    per_example_kl = torch.sum(
        labels * (torch.log(labels + 1e-12) - log_probs),
        dim=1
    )

    loss = (per_example_kl * weights).mean()

    return loss

In [31]:
def distribution_metrics(probs, labels):
    eps = 1e-12

    kl = np.sum(
        labels * (np.log(labels + eps) - np.log(probs + eps)),
        axis=1
    ).mean()

    m = 0.5 * (labels + probs)

    js = 0.5 * (
        np.sum(labels * (np.log(labels + eps) - np.log(m + eps)), axis=1)
        +
        np.sum(probs * (np.log(probs + eps) - np.log(m + eps)), axis=1)
    ).mean()

    hard_accuracy = (probs.argmax(axis=1) == labels.argmax(axis=1)).mean()

    pred_entropy = -np.sum(probs * np.log2(probs + eps), axis=1)
    gold_entropy = -np.sum(labels * np.log2(labels + eps), axis=1)

    if np.std(pred_entropy) == 0 or np.std(gold_entropy) == 0:
        entropy_corr = np.nan
    else:
        entropy_corr = np.corrcoef(pred_entropy, gold_entropy)[0, 1]

    entropy_mae = np.mean(np.abs(pred_entropy - gold_entropy))

    return {
        "kl_divergence": float(kl),
        "js_divergence": float(js),
        "hard_accuracy": float(hard_accuracy),
        "entropy_correlation": float(entropy_corr),
        "entropy_mae": float(entropy_mae)
    }

In [32]:
def predict_loader(model, loader):
    model.eval()

    all_probs = []
    all_labels = []
    losses = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            labels = batch["labels"].float()
            loss = compute_loss(model, batch)

            logits = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )

            probs = torch.nn.functional.softmax(logits, dim=-1)

            losses.append(loss.item())
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    probs = np.vstack(all_probs)
    labels = np.vstack(all_labels)

    return probs, labels, float(np.mean(losses))

In [33]:
def evaluate_overall(model, loader):
    probs, labels, loss = predict_loader(model, loader)
    metrics = distribution_metrics(probs, labels)
    metrics["eval_loss"] = loss
    return metrics

In [34]:
def evaluate_by_entropy_group(model, loader, original_df):
    probs, labels, _ = predict_loader(model, loader)

    out = original_df.reset_index(drop=True).copy()
    out["pred_0"] = probs[:, 0]
    out["pred_1"] = probs[:, 1]
    out["pred_2"] = probs[:, 2]
    out["gold_0"] = labels[:, 0]
    out["gold_1"] = labels[:, 1]
    out["gold_2"] = labels[:, 2]

    rows = []

    for group_name, group_df in out.groupby("entropy_group"):
        group_probs = group_df[["pred_0", "pred_1", "pred_2"]].values
        group_labels = group_df[["gold_0", "gold_1", "gold_2"]].values

        metrics = distribution_metrics(group_probs, group_labels)

        row = {
            "entropy_group": group_name,
            "n": len(group_df)
        }
        row.update(metrics)
        rows.append(row)

    return pd.DataFrame(rows).sort_values("entropy_group"), out

In [35]:
LEARNING_RATE = 5e-6
WEIGHT_DECAY = 0.10
NUM_EPOCHS = 10
PATIENCE = 3

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [36]:
output_dir = "../models/roberta_entropy_aware_hatespeech_lewidi_2"
best_model_dir = os.path.join(output_dir, "best_model")

os.makedirs(best_model_dir, exist_ok=True)

history_rows = []

best_val_js = float("inf")
epochs_without_improvement = 0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()

    train_losses = []
    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")

    for batch in progress:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()

        loss = compute_loss(model, batch)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=1.0
        )

        optimizer.step()

        train_losses.append(loss.item())
        progress.set_postfix({"train_loss": np.mean(train_losses)})

    val_metrics = evaluate_overall(model, val_loader)
    val_group_metrics, _ = evaluate_by_entropy_group(model, val_loader, val_df)

    print("\n" + "=" * 80)
    print(f"Epoch {epoch}")
    print(f"Train loss: {np.mean(train_losses):.6f}")
    print("Validation overall:")
    for k, v in val_metrics.items():
        print(f"{k}: {v}")
    print("\nValidation by entropy group:")
    print(val_group_metrics.to_string(index=False))
    print("=" * 80 + "\n")

    history_row = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
    }

    for k, v in val_metrics.items():
        history_row[f"val_{k}"] = v

    history_rows.append(history_row)

    current_val_js = val_metrics["js_divergence"]

    if current_val_js < best_val_js:
        best_val_js = current_val_js
        epochs_without_improvement = 0

        torch.save(model.state_dict(), os.path.join(best_model_dir, "pytorch_model.bin"))
        tokenizer.save_pretrained(best_model_dir)

        print(f"New best model saved. Best validation JS: {best_val_js:.6f}")

    else:
        epochs_without_improvement += 1
        print(f"No improvement. Patience: {epochs_without_improvement}/{PATIENCE}")

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break

    torch.cuda.empty_cache()

Epoch 1/10:   0%|          | 0/930 [00:00<?, ?it/s]


Epoch 1
Train loss: 0.411102
Validation overall:
kl_divergence: 0.37321123480796814
js_divergence: 0.10404332727193832
hard_accuracy: 0.7957526545908807
entropy_correlation: 0.41655853741916077
entropy_mae: 0.6199510097503662
eval_loss: 0.4024918969019805

Validation by entropy group:
 entropy_group   n  kl_divergence  js_divergence  hard_accuracy  entropy_correlation  entropy_mae
  high_entropy  57       0.251882       0.054665       0.561404             0.047398     0.272044
   low_entropy 277       0.345933       0.091380       0.750903             0.049730     0.427872
medium_entropy 344       0.513396       0.127327       0.549419                  NaN     0.347097
  zero_entropy 923       0.336644       0.102215       0.915493                  NaN     0.800773

New best model saved. Best validation JS: 0.104043


Epoch 2/10:   0%|          | 0/930 [00:00<?, ?it/s]


Epoch 2
Train loss: 0.389943
Validation overall:
kl_divergence: 0.3824964463710785
js_divergence: 0.10191229730844498
hard_accuracy: 0.8076202373516552
entropy_correlation: 0.41757173781107476
entropy_mae: 0.57728111743927
eval_loss: 0.42305149388785407

Validation by entropy group:
 entropy_group   n  kl_divergence  js_divergence  hard_accuracy  entropy_correlation  entropy_mae
  high_entropy  57       0.328224       0.064921       0.578947             0.040415     0.322294
   low_entropy 277       0.382623       0.095866       0.725632             0.042020     0.407678
medium_entropy 344       0.560811       0.131290       0.651163                  NaN     0.355358
  zero_entropy 923       0.319353       0.095062       0.904659                  NaN     0.726637

New best model saved. Best validation JS: 0.101912


Epoch 3/10:   0%|          | 0/930 [00:00<?, ?it/s]

...


Epoch 4
Train loss: 0.356091
Validation overall:
kl_divergence: 0.3847792446613312
js_divergence: 0.10407757759094238
hard_accuracy: 0.7776389756402249
entropy_correlation: 0.40471452563049465
entropy_mae: 0.5862532258033752
eval_loss: 0.42064931457585625

Validation by entropy group:
 entropy_group   n  kl_divergence  js_divergence  hard_accuracy  entropy_correlation  entropy_mae
  high_entropy  57       0.293699       0.060858       0.508772             0.122474     0.316622
   low_entropy 277       0.369908       0.093658       0.736462             0.097308     0.390770
medium_entropy 344       0.555607       0.131986       0.468023                  NaN     0.312686
  zero_entropy 923       0.331200       0.099472       0.921993                  NaN     0.763529

No improvement. Patience: 1/3


Epoch 5/10:   0%|          | 0/930 [00:00<?, ?it/s]


Epoch 5
Train loss: 0.336646
Validation overall:
kl_divergence: 0.4008272886276245
js_divergence: 0.09866229444742203
hard_accuracy: 0.7845096814490943
entropy_correlation: 0.40604324914529516
entropy_mae: 0.5182065367698669
eval_loss: 0.4598547474730133

Validation by entropy group:
 entropy_group   n  kl_divergence  js_divergence  hard_accuracy  entropy_correlation  entropy_mae
  high_entropy  57       0.441752       0.079812       0.631579             0.143187     0.416113
   low_entropy 277       0.426040       0.096730       0.722022             0.079036     0.385073
medium_entropy 344       0.660171       0.138691       0.505814                  NaN     0.325348
  zero_entropy 923       0.294077       0.085488       0.916576                  NaN     0.636344

No improvement. Patience: 2/3


Epoch 6/10:   0%|          | 0/930 [00:00<?, ?it/s]


Epoch 6
Train loss: 0.312610
Validation overall:
kl_divergence: 0.44918662309646606
js_divergence: 0.10277272015810013
hard_accuracy: 0.7682698313554028
entropy_correlation: 0.37623480137633336
entropy_mae: 0.4982146918773651
eval_loss: 0.5241658813735046

Validation by entropy group:
 entropy_group   n  kl_divergence  js_divergence  hard_accuracy  entropy_correlation  entropy_mae
  high_entropy  57       0.507147       0.088569       0.578947             0.141372     0.471173
   low_entropy 277       0.494343       0.104037       0.707581             0.072605     0.382444
medium_entropy 344       0.782808       0.147799       0.500000                  NaN     0.352300
  zero_entropy 923       0.307716       0.086489       0.898158                  NaN     0.589011

No improvement. Patience: 3/3
Early stopping triggered.


In [37]:
model.load_state_dict(
    torch.load(os.path.join(best_model_dir, "pytorch_model.bin"), map_location=device)
)
model.to(device)

val_results = evaluate_overall(model, val_loader)
test_results = evaluate_overall(model, test_loader)

val_group_results, val_pred_df = evaluate_by_entropy_group(model, val_loader, val_df)
test_group_results, test_pred_df = evaluate_by_entropy_group(model, test_loader, test_df)

print("Final validation overall:")
print(val_results)

print("\nFinal validation by entropy group:")
display(val_group_results)

print("\nFinal test overall:")
print(test_results)

print("\nFinal test by entropy group:")
display(test_group_results)

/tmp/ipykernel_11481/156096654.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(best_model_dir, "pytorch_model.bin"), map_location=device)


Final validation overall:
{'kl_divergence': 0.3823968768119812, 'js_divergence': 0.09810124337673187, 'hard_accuracy': 0.7907557776389756, 'entropy_correlation': 0.41754934911093455, 'entropy_mae': 0.5370948314666748, 'eval_loss': 0.4324894678769725}

Final validation by entropy group:


,entropy_group,n,kl_divergence,js_divergence,hard_accuracy,entropy_correlation,entropy_mae
0,high_entropy,57,0.361946,0.071568,0.578947,0.073375,0.366937
1,low_entropy,277,0.388198,0.092353,0.754513,0.059948,0.386685
2,medium_entropy,344,0.609979,0.135351,0.508721,NaN,0.331341
3,zero_entropy,923,0.297100,0.087582,0.919827,NaN,0.669426



Final test overall:
{'kl_divergence': 0.4037986099720001, 'js_divergence': 0.10362039506435394, 'hard_accuracy': 0.7670206121174266, 'entropy_correlation': 0.38153544657445443, 'entropy_mae': 0.5582501292228699, 'eval_loss': 0.4622290988664816}

Final test by entropy group:


,entropy_group,n,kl_divergence,js_divergence,hard_accuracy,entropy_correlation,entropy_mae
0,high_entropy,66,0.398489,0.075128,0.515152,-0.020914,0.427405
1,low_entropy,276,0.389864,0.092805,0.688406,0.163179,0.382384
2,medium_entropy,331,0.595352,0.133648,0.534743,NaN,0.332935
3,zero_entropy,928,0.339997,0.098153,0.891164,NaN,0.700227


In [38]:
pyevall_dir = "../results/pyevall/entropy_aware_hatespeech_lewidi_2"
os.makedirs(pyevall_dir, exist_ok=True)

gold_records = []
pred_records = []

for _, row in test_pred_df.iterrows():
    record_id = str(row["comment_id"])

    gold_records.append({
        "test_case": "hatespeech",
        "id": record_id,
        "value": {
            "0": float(row["gold_0"]),
            "1": float(row["gold_1"]),
            "2": float(row["gold_2"])
        }
    })

    pred_records.append({
        "test_case": "hatespeech",
        "id": record_id,
        "value": {
            "0": float(row["pred_0"]),
            "1": float(row["pred_1"]),
            "2": float(row["pred_2"])
        }
    })

gold_path = os.path.join(pyevall_dir, "hatespeech_gold.json")
pred_path = os.path.join(pyevall_dir, "hatespeech_pred.json")

with open(gold_path, "w", encoding="utf-8") as f:
    json.dump(gold_records, f, indent=2)

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(pred_records, f, indent=2)

print("Gold:", gold_path)
print("Pred:", pred_path)

Gold: ../results/pyevall/entropy_aware_hatespeech_lewidi_2/hatespeech_gold.json
Pred: ../results/pyevall/entropy_aware_hatespeech_lewidi_2/hatespeech_pred.json


In [39]:
from pyevall.evaluation import PyEvALLEvaluation
from pyevall.utils.utils import PyEvALLUtils

params = {
    PyEvALLUtils.PARAM_REPORT: PyEvALLUtils.PARAM_OPTION_REPORT_EMBEDDED
}

evaluator = PyEvALLEvaluation()

pyevall_report = evaluator.evaluate(
    pred_path,
    gold_path,
    [
        "CrossEntropy",
        "ICMSoft",
        "ICMSoftNorm"
    ],
    **params
)

pyevall_data = pyevall_report.report if hasattr(pyevall_report, "report") else pyevall_report.__dict__
pyevall_data

2026-06-02 08:41:52,816 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['CrossEntropy', 'ICMSoft', 'ICMSoftNorm']
2026-06-02 08:41:53,158 - pyevall.metrics.metrics - INFO -             evaluate() - Executing Cross Entropy evaluation method for testcase hatespeech
2026-06-02 08:41:53,582 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft evaluation method for testcase hatespeech
2026-06-02 08:41:55,089 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM-Soft Normalized evaluation method for testcase hatespeech
2026-06-02 08:41:55,091 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft evaluation method for testcase hatespeech
2026-06-02 08:41:55,812 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft evaluation method for testcase hatespeech
cargado 46


{'metrics': {'CrossEntropy': {'name': 'Cross Entropy',
   'acronym': 'CE',
   'description': 'Coming soon!',
   'status': 'OK',
   'results': {'test_cases': [{'name': 'hatespeech',
      'average': 1.0109927081716288}],
    'average_per_test_case': 1.0109927081716288}},
  'ICMSoft': {'name': 'Information Contrast Model Soft',
   'acronym': 'ICM-Soft',
   'description': 'Coming soon!',
   'status': 'OK',
   'results': {'test_cases': [{'name': 'hatespeech',
      'average': -2.2320832944349305}],
    'average_per_test_case': -2.2320832944349305}},
  'ICMSoftNorm': {'name': 'Normalized Information Contrast Model Soft',
   'acronym': 'ICM-Soft-Norm',
   'description': 'Coming soon!',
   'status': 'OK',
   'results': {'test_cases': [{'name': 'hatespeech',
      'average': 0.2641746200618378}],
    'average_per_test_case': 0.2641746200618378}}},
 'files': {'hatespeech_pred.json': {'name': 'hatespeech_pred.json',
   'status': 'OK',
   'gold': False,
   'description': 'The file is correctly pa

In [40]:
pyevall_summary = {}

for metric_name, metric_data in pyevall_data["metrics"].items():
    pyevall_summary[metric_name] = metric_data["results"]["average_per_test_case"]

pyevall_summary

{'CrossEntropy': 1.0109927081716288,
 'ICMSoft': -2.2320832944349305,
 'ICMSoftNorm': 0.2641746200618378}

In [41]:
os.makedirs("../results/tables", exist_ok=True)

results_path = "../results/tables/roberta_entropy_aware_hatespeech_lewidi_2_results.txt"
history_path = "../results/tables/roberta_entropy_aware_hatespeech_lewidi_2_training_history.csv"
test_pred_path = "../results/tables/roberta_entropy_aware_hatespeech_lewidi_2_test_predictions.csv"
test_group_path = "../results/tables/roberta_entropy_aware_hatespeech_lewidi_2_test_entropy_groups.csv"

history_df = pd.DataFrame(history_rows)

history_df.to_csv(history_path, index=False)
test_pred_df.to_csv(test_pred_path, index=False)
test_group_results.to_csv(test_group_path, index=False)

with open(results_path, "w", encoding="utf-8") as f:
    f.write("ROBERTA ENTROPY-AWARE HATESPEECH LEWIDI BASELINE RESULTS\n")
    f.write("=" * 90 + "\n\n")

    f.write("1. RUN INFORMATION\n")
    f.write("-" * 90 + "\n")
    f.write(f"Run timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Model name: {model_name}\n")
    f.write(f"Output directory: {output_dir}\n")
    f.write(f"Best validation JS: {best_val_js}\n")
    f.write(f"Epochs completed: {len(history_df)}\n\n")

    f.write("2. DATASET STRATEGY\n")
    f.write("-" * 90 + "\n")
    f.write("Target: hatespeech annotator label distribution only.\n")
    f.write(f"Minimum annotations per comment: {MIN_ANNOTATIONS}\n")
    f.write(f"Original comments: {len(comment_df)}\n")
    f.write(f"Reliable comments: {len(reliable_df)}\n")
    f.write(f"Percent kept: {round(len(reliable_df) / len(comment_df) * 100, 2)}%\n\n")

    f.write("Full split counts:\n")
    f.write(full_split_counts.to_string(index=False))
    f.write("\n\nReliable subset split counts:\n")
    f.write(reliable_split_counts.to_string(index=False))
    f.write("\n\nEntropy group summary:\n")
    f.write(entropy_group_summary.to_string(index=False))
    f.write("\n\n")

    f.write("3. MODEL OBJECTIVE\n")
    f.write("-" * 90 + "\n")
    f.write("Architecture: RoBERTa encoder with one 3-class distribution head.\n")
    f.write("Loss: KL divergence between predicted and gold annotator distributions.\n")
    f.write("Validation: overall and entropy-sliced JS/KL/accuracy/entropy correlation.\n\n")

    f.write("4. TRAINING SETUP\n")
    f.write("-" * 90 + "\n")
    f.write(f"Max length: {MAX_LENGTH}\n")
    f.write(f"Train batch size: {BATCH_SIZE}\n")
    f.write(f"Eval batch size: {EVAL_BATCH_SIZE}\n")
    f.write(f"Learning rate: {LEARNING_RATE}\n")
    f.write(f"Weight decay: {WEIGHT_DECAY}\n")
    f.write("Dropout: 0.35\n")
    f.write("Frozen lower RoBERTa layers: 4\n")
    f.write(f"Max epochs: {NUM_EPOCHS}\n")
    f.write(f"Early stopping patience: {PATIENCE}\n\n")

    f.write("5. TRAINING HISTORY\n")
    f.write("-" * 90 + "\n")
    f.write(history_df.to_string(index=False))
    f.write("\n\n")

    f.write("6. FINAL VALIDATION RESULTS\n")
    f.write("-" * 90 + "\n")
    for k, v in val_results.items():
        f.write(f"{k}: {v}\n")
    f.write("\nValidation by entropy group:\n")
    f.write(val_group_results.to_string(index=False))
    f.write("\n\n")

    f.write("7. FINAL TEST RESULTS\n")
    f.write("-" * 90 + "\n")
    for k, v in test_results.items():
        f.write(f"{k}: {v}\n")
    f.write("\nTest by entropy group:\n")
    f.write(test_group_results.to_string(index=False))
    f.write("\n\n")

    f.write("8. PYEVAL LEWIDI RESULTS\n")
    f.write("-" * 90 + "\n")
    for k, v in pyevall_summary.items():
        f.write(f"{k}: {v}\n")
    f.write("\n")

   

print("Saved:", results_path)
print(open(results_path, encoding="utf-8").read())

Saved: ../results/tables/roberta_entropy_aware_hatespeech_lewidi_2_results.txt
ROBERTA ENTROPY-AWARE HATESPEECH LEWIDI BASELINE RESULTS

1. RUN INFORMATION
------------------------------------------------------------------------------------------
Run timestamp: 2026-06-02 08:41:56
Model name: roberta-base
Output directory: ../models/roberta_entropy_aware_hatespeech_lewidi_2
Best validation JS: 0.09810124337673187
Epochs completed: 6

2. DATASET STRATEGY
------------------------------------------------------------------------------------------
Target: hatespeech annotator label distribution only.
Minimum annotations per comment: 2
Original comments: 19761
Reliable comments: 10639
Percent kept: 53.84%

Full split counts:
     split  comments  mean_annotation_count  mean_entropy  zero_entropy_percent
      test      2965               2.601349      0.228469                 77.30
     train     13832               2.487927      0.223532                 77.56
validation      2964           